# Data Splitting into Train/Val/Test Sets

This notebook splits the preprocessed data into training, validation, and test sets.

**Splitting Strategy**: Data is distributed per corrosion inhibitor to test the model's ability to generalize to new, unseen inhibitor types while maintaining proper train/val/test separation.

**Key Principles**:
1. All 4 images of each inhibitor sample must be in the same split
2. Different inhibitors go to different splits (except NaCl, which stays in training)
3. No data leakage between sets (no sample appears in multiple splits)

In [1]:
"""
Import required libraries for data splitting
"""
from pathlib import Path
import os
import random
import shutil

from ordered_set import OrderedSet
import pandas as pd

In [2]:
"""
Configure splitting parameters and paths
"""
# Number of samples in each split
num_val = 25      # Number of validation sample IDs
num_test = 25     # Number of test sample IDs
# Remaining samples go to training set

# Random seed for reproducible splitting
RANDOM_SEED = 5

# Input data paths
original_path = Path("../rawdata/unet")
inhibitor_list_file = Path("../rawdata/processed/volume-loss/volume-loss.csv")

# Output data paths (splits created here)
save_path = Path("../../data/")

# Subdirectory naming
before_path = "before"
after_path = "after"
image_path = "image"
height_path = "height"

In [3]:
"""
Extract inhibitor information and identify NaCl samples
"""
# Load the volume loss and inhibitor metadata
inhibitors = pd.read_csv(inhibitor_list_file)

# Filter for all NaCl samples (these will be kept in training set)
# We want to test generalization to new inhibitor types, so NaCl samples
# (a common baseline) are always included in training
nacl_samples = inhibitors[inhibitors["name"].str.contains("NaCl", case=False)].id

# Extract unique sample base IDs (e.g., "x-x-x" from "x-x-x-1")
nacl_subid = OrderedSet([i[:-2] for i in nacl_samples])

print(f"Found {len(nacl_subid)} unique NaCl sample bases")
print(f"Total unique images: {len(nacl_samples)}")

Found 52 unique NaCl sample bases
Total unique images: 208


In [4]:
"""
Perform data splitting with balanced inhibitor distribution
"""
# Get the unique sample base IDs from the actual data
total_sub_ids = OrderedSet([i[:-6] for i in os.listdir(original_path / before_path / image_path)])

# Get all NaCl samples contained in the data
contained_nacl_subid = nacl_subid - (nacl_subid - total_sub_ids)

# Remove the NaCl samples from the pool to split
sub_ids = list(total_sub_ids - contained_nacl_subid)

# Shuffle for random split (with reproducible seed)
random.Random(RANDOM_SEED).shuffle(sub_ids)

# Split non-NaCl samples
test_val_ids = sub_ids[:(num_val + num_test)]
train_ids = sub_ids[(num_val + num_test):]

# Further split test_val into val and test
val_ids = test_val_ids[:num_val]
test_ids = test_val_ids[num_val:]

# Add NaCl samples back to training data
train_ids += list(contained_nacl_subid)

print(f"Training set: {len(train_ids)} samples ({len(contained_nacl_subid)} are NaCl)")
print(f"Validation set: {len(val_ids)} samples")
print(f"Test set: {len(test_ids)} samples")
print(f"Total: {len(train_ids) + len(val_ids) + len(test_ids)} samples")

Training set: 161 samples (36 are NaCl)
Validation set: 25 samples
Test set: 25 samples
Total: 211 samples


In [ ]:
"""
Verify that known problematic samples are in the correct set
"""
# Samples with known data quality issues that should be in training set
problematic_samples = ["3-8-3-", "3-8-6-", "4-6-10-", "2-5-10-", "2-5-11-"]

for sample_pattern in problematic_samples:
    try:
        idx = [s for s in train_ids if s.startswith(sample_pattern)][0]
        print(f"{sample_pattern}* is in training set")
    except IndexError:
        print(f"WARNING: {sample_pattern}* not found in training set!")

(2, 149, 112, 17, 119)

In [ ]:
def copy_files(id_list: list, target: str) -> None:
    """
    Copy data files for given sample IDs to a target split directory.

    Copies all 4 images, height profiles, and masks for each sample ID
    into the appropriate train/val/test directory structure.

    Args:
        id_list: List of sample base IDs (e.g., ["x-x-x", ...])
        target: Target directory name ("train", "val", or "test")
    """
    # Copy images and heights for both before and after
    for time in [before_path, after_path]:
        for data_type in [image_path, height_path]:
            source = original_path / time / data_type
            target_dir = save_path / target / time / data_type

            # Remove already existing folder to ensure clean split
            try:
                shutil.rmtree(target_dir)
            except FileNotFoundError:
                pass

            # Create target directory
            os.makedirs(target_dir, exist_ok=True)

            # Copy all files matching the sample IDs
            for sample_id in id_list:
                for file in source.rglob(f"{sample_id}*"):
                    shutil.copy(file, target_dir)

    # Copy masks (one mask per sample, in shared directory)
    mask_target_dir = save_path / target / "mask"

    # Remove already existing mask folder
    try:
        shutil.rmtree(mask_target_dir)
    except FileNotFoundError:
        pass

    # Create mask directory
    os.makedirs(mask_target_dir, exist_ok=True)

    # Copy all relevant masks
    for sample_id in id_list:
        source = original_path / "mask"
        for file in source.rglob(f"{sample_id}*"):
            shutil.copy(file, mask_target_dir)

In [ ]:
"""
Execute the data split: copy files to train, val, and test directories
"""
print("Copying validation set...")
copy_files(val_ids, "val")

print("Copying test set...")
copy_files(test_ids, "test")

print("Copying training set...")
copy_files(train_ids, "train")

print("Data splitting complete!")

In [ ]:
"""
Verify data splitting: ensure no data leakage between sets
"""
# Get all files in each split
train_path = save_path / "train"
val_path = save_path / "val"
test_path = save_path / "test"

train_files = OrderedSet([p.name for p in train_path.rglob("*.*")])
val_files = OrderedSet([p.name for p in val_path.rglob("*.*")])
test_files = OrderedSet([p.name for p in test_path.rglob("*.*")])

# Check for overlaps (data leakage)
train_val_overlap = train_files.intersection(val_files)
train_test_overlap = train_files.intersection(test_files)
val_test_overlap = test_files.intersection(val_files)

if not (train_val_overlap or train_test_overlap or val_test_overlap):
    print("Data splitting successful! No data leakage detected.")
    print(f"  - Train: {len(train_files)} files")
    print(f"  - Val: {len(val_files)} files")
    print(f"  - Test: {len(test_files)} files")
else:
    print("ERROR: Data leakage detected!")
    if train_val_overlap:
        print(f"  - Train/Val overlap: {len(train_val_overlap)} files")
    if train_test_overlap:
        print(f"  - Train/Test overlap: {len(train_test_overlap)} files")
    if val_test_overlap:
        print(f"  - Val/Test overlap: {len(val_test_overlap)} files")

(OrderedSet(), OrderedSet(), OrderedSet())